# B8：Agent 統計評估框架 — 如何量化 AI 品質

> **場景：** 台灣律師事務所部署合約審查 Agent（每月審查 5,000 份合約）。  
> 法遵部門要求：(1) 上線前提交量化準確率報告，(2) 每次模型更新後 24 小時內完成驗證，(3) 能持續監控線上品質（偵測 model drift）。  
> 傳統軟體測試方法無法評估 LLM 輸出的「品質」。

## 這本 Notebook 要回答的核心問題

每個架構決策都會問：**「為什麼用 X，不用 Y？」**

| 決策 | 選擇 | 拒絕的方案 |
|------|------|------------|
| 主要評估方式 | LLM-as-Judge + 規則驗證 | 只靠人工評估 |
| 測試集管理 | Golden Dataset 版本控制（git-tracked） | 每次手動挑測試案例 |
| 線上評估 | Shadow Mode（離線評估新模型） | 直接上線 A/B 測試 |
| 評估指標 | LLM-native + Task-specific metrics | 只看 BLEU/ROUGE |
| 迴歸測試 | CI/CD 觸發（PR merge 前自動跑） | 手動執行 |

In [ ]:
import json
import math
import random
import hashlib
import statistics
import time
from dataclasses import dataclass, field
from typing import Optional
from datetime import datetime

try:
    from dotenv import load_dotenv; load_dotenv()
    from langchain_openai import ChatOpenAI
    from langchain_core.messages import HumanMessage, SystemMessage
    llm_judge = ChatOpenAI(model="gpt-4o", temperature=0)    # 模擬 Gemini Pro 作為 Judge
    llm_fast  = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    LLM_AVAILABLE = True
    print("✅ LLM 可用（使用 GPT-4o 模擬 Gemini Pro Judge）")
except Exception:
    LLM_AVAILABLE = False
    print("⚠️  LLM 未設定，使用模擬輸出")

random.seed(42)  # 可重現的模擬結果

print("\n場景：台灣律師事務所合約審查 Agent")
print("每月審查合約：5,000 份 | 法遵要求：量化準確率 + 24 小時驗證")

## 評估管線架構全貌

```
                    ┌─────────────────────────────┐
                    │     評估管線架構              │
                    └─────────────────────────────┘

Golden Dataset ──→ [Offline Evaluator] ──→ Eval Report
                          │
                          ├── LLM-as-Judge (品質)
                          ├── Rule Validator (結構)
                          └── LLM-native Metrics

Production Traffic ──→ [Shadow Mode Runner] ──→ A/B Comparison
                              │
                              ├── Current Model
                              └── Candidate Model (silent)

CI/CD Pipeline ──→ [Regression Gate] ──→ Block/Allow merge
                (accuracy drop > 5% = block)
```

**三層評估策略**

| 層級 | 觸發時機 | 目的 | 時間成本 |
|------|----------|------|----------|
| Offline Eval | 開發階段 / PR | 驗證模型品質 | 1-2 小時 |
| Shadow Mode | 新模型上線前 | 對比現有模型 | 1-3 天 |
| Drift Monitor | 持續線上 | 偵測品質下滑 | 即時 |

## 決策 1：LLM-as-Judge vs. 人工評估

**❌ 被拒絕的方案：純人工評估**  
律師評審每份合約審查結果平均 30 分鐘，5,000 份 × 30 分鐘 = **2,500 小時/月**。  
即使按 6 位律師全職計算（每月 160 小時/人），需要 **15.6 人月** 才能做完評估，完全不可行。  
此外，不同評審律師的判斷標準不一致（inter-rater reliability 通常只有 0.65-0.75）。

**✅ 選擇的方案：LLM-as-Judge + 規則驗證**
- 用 GPT-4o 或 Gemini Pro 作為評審 Judge，與人工評估的相關性 > 0.85
- 人工評審只用於建立 Golden Dataset（一次性），之後 Judge 自動跑
- 規則驗證確保結構性要求（如：必須包含合約雙方名稱、日期、條款編號）

**Judge Prompt 設計關鍵：**
1. 要求結構化輸出（score + reasoning + specific_issues），避免模糊評分
2. 隨機化答案順序，避免 position bias（Judge 偏向先看到的答案）
3. 用不同 LLM 作為 Judge（避免自我偏袒：不用同一個模型評估自己的輸出）

In [ ]:
@dataclass
class JudgeResult:
    score: float          # 0.0 - 1.0
    reasoning: str
    specific_issues: list[str]
    clause_coverage: float   # 關鍵條款覆蓋率
    risk_accuracy: float     # 風險識別準確性
    hallucination_flag: bool # 是否有捏造內容


class LLMJudge:
    """
    LLM-as-Judge：用 GPT-4o / Gemini Pro 評估合約審查品質

    設計原則：
    1. Judge 與被評估的模型不同（避免自我偏袒）
    2. 隨機化 Agent 輸出順序（避免 position bias）
    3. 要求結構化輸出（score + reasoning + issues）
    4. 與人工標籤對照，持續校正 Judge 的準確性
    """

    JUDGE_SYSTEM_PROMPT = """你是一位資深合約律師，負責評估 AI 合約審查系統的輸出品質。
請客觀評估以下合約審查結果，重點檢查：
1. 關鍵條款是否都有識別（付款條件、違約責任、終止條款、智財權）
2. 風險評估是否準確（高/中/低風險分類）
3. 是否有捏造不存在的條款（幻覺）
4. 建議是否具體可行

你的評估必須嚴格、客觀，不受答案呈現順序影響。"""

    def _build_judge_prompt(self, contract_excerpt: str, agent_output: str,
                             reference_output: str, randomize: bool = True) -> str:
        """建立 Judge Prompt，可選擇隨機化輸出順序（避免 position bias）"""
        outputs = [
            ("A", agent_output, "待評估"),
            ("B", reference_output, "參考"),
        ]
        if randomize and random.random() > 0.5:
            outputs = [("A", reference_output, "參考"), ("B", agent_output, "待評估")]
            self._agent_is_b = True
        else:
            self._agent_is_b = False

        return f"""合約節錄：
{contract_excerpt}

審查結果 A：
{outputs[0][1]}

審查結果 B：
{outputs[1][1]}

請評估「審查結果 A」的品質（0-10 分），並以 JSON 格式回覆：
{{
  "score": <0-10 的整數>,
  "reasoning": "<整體評估說明>",
  "specific_issues": ["<問題1>", "<問題2>"],
  "clause_coverage": <0.0-1.0，關鍵條款覆蓋比例>,
  "risk_accuracy": <0.0-1.0，風險識別準確性>,
  "hallucination_flag": <true/false，是否有捏造內容>
}}"""

    async def judge_contract_review(
        self,
        contract_excerpt: str,
        agent_output: str,
        reference_output: str,
    ) -> JudgeResult:
        prompt = self._build_judge_prompt(contract_excerpt, agent_output, reference_output)

        if LLM_AVAILABLE:
            response = await llm_judge.ainvoke([
                SystemMessage(content=self.JUDGE_SYSTEM_PROMPT),
                HumanMessage(content=prompt),
            ])
            try:
                raw = response.content.strip()
                if "```" in raw:
                    raw = raw.split("```")[1].replace("json", "").strip()
                data = json.loads(raw)
            except Exception:
                data = {"score": 7, "reasoning": "解析失敗",
                        "specific_issues": [], "clause_coverage": 0.8,
                        "risk_accuracy": 0.75, "hallucination_flag": False}
        else:
            # 模擬 Judge 輸出（基於關鍵詞啟發式）
            has_key_clauses = all(kw in agent_output for kw in ["付款", "違約", "終止"])
            score = random.uniform(7.5, 9.2) if has_key_clauses else random.uniform(4.5, 6.5)
            issues = [] if has_key_clauses else ["缺少違約責任條款分析", "未識別自動續約風險"]
            data = {
                "score": round(score, 1),
                "reasoning": "合約審查覆蓋了主要條款" if has_key_clauses else "遺漏部分關鍵條款",
                "specific_issues": issues,
                "clause_coverage": 0.88 if has_key_clauses else 0.62,
                "risk_accuracy": 0.85 if has_key_clauses else 0.60,
                "hallucination_flag": False,
            }

        return JudgeResult(
            score=data["score"] / 10.0,
            reasoning=data["reasoning"],
            specific_issues=data.get("specific_issues", []),
            clause_coverage=data["clause_coverage"],
            risk_accuracy=data["risk_accuracy"],
            hallucination_flag=data["hallucination_flag"],
        )


def compute_inter_rater_reliability(judge_scores: list[float],
                                     human_scores: list[float]) -> dict:
    """
    計算 Judge 與人工評審的一致性（Cohen's Kappa 近似）
    目標：相關係數 > 0.85 才信任 LLM-as-Judge
    """
    n = len(judge_scores)
    assert n == len(human_scores) and n > 0

    # Pearson 相關係數
    mean_j = sum(judge_scores) / n
    mean_h = sum(human_scores) / n
    cov = sum((j - mean_j) * (h - mean_h) for j, h in zip(judge_scores, human_scores)) / n
    std_j = math.sqrt(sum((j - mean_j) ** 2 for j in judge_scores) / n)
    std_h = math.sqrt(sum((h - mean_h) ** 2 for h in human_scores) / n)
    pearson_r = cov / (std_j * std_h) if (std_j * std_h) > 0 else 0.0

    # 平均絕對誤差
    mae = sum(abs(j - h) for j, h in zip(judge_scores, human_scores)) / n

    # 方向一致率（兩者都認為「好」或都認為「差」）
    threshold = 0.7
    direction_agree = sum(
        1 for j, h in zip(judge_scores, human_scores)
        if (j >= threshold) == (h >= threshold)
    ) / n

    return {
        "pearson_r": round(pearson_r, 3),
        "mae": round(mae, 3),
        "direction_agreement": round(direction_agree, 3),
        "trust_judge": pearson_r > 0.85,
    }


# ── 示範 ──────────────────────────────────────────────────────
import asyncio

CONTRACT_EXCERPT = """第3條 付款條件：買方應於簽約後30日內支付頭期款新台幣100萬元。
第7條 違約責任：任一方違約應賠償他方合約總金額20%之違約金。
第12條 合約終止：雙方得於30日書面通知後終止合約。"""

GOOD_AGENT_OUTPUT = """【合約審查報告】
■ 付款條件（第3條）：頭期款100萬，30日付款期限，屬標準商業條件。
■ 違約責任（第7條）：違約金20%，建議確認是否符合貴公司風險承受度。
■ 終止條款（第12條）：30日通知期，合理。
■ 整體風險等級：中等。"""

WEAK_AGENT_OUTPUT = """本合約看起來還不錯，條款算完整。
付款30天沒問題，終止條款也有寫。整體建議簽署。"""

REFERENCE_OUTPUT = """【參考標準審查】
■ 付款條件：符合市場慣例，但建議加入逾期付款利息條款。
■ 違約責任：20% 違約金屬中等水準，與業界相符。
■ 終止條款：需確認「重大違約」的定義範圍。
■ 缺少：智慧財產權歸屬、保密條款、準據法條款。
■ 整體風險：中等偏高。"""

judge = LLMJudge()

async def run_judge_demo():
    print("LLM-as-Judge 評估示範：")
    print("=" * 55)

    good_result = await judge.judge_contract_review(
        CONTRACT_EXCERPT, GOOD_AGENT_OUTPUT, REFERENCE_OUTPUT)
    weak_result = await judge.judge_contract_review(
        CONTRACT_EXCERPT, WEAK_AGENT_OUTPUT, REFERENCE_OUTPUT)

    for label, result in [("完整審查", good_result), ("簡略審查", weak_result)]:
        print(f"\n[{label}]")
        print(f"  整體分數：{result.score:.2f} / 1.00")
        print(f"  條款覆蓋率：{result.clause_coverage:.0%}")
        print(f"  風險識別準確性：{result.risk_accuracy:.0%}")
        print(f"  幻覺檢測：{'⚠️  發現幻覺' if result.hallucination_flag else '✅ 無幻覺'}")
        if result.specific_issues:
            print(f"  具體問題：{result.specific_issues}")

    # 模擬 30 筆人工 vs Judge 比較
    n_samples = 30
    human_scores  = [round(random.uniform(0.6, 1.0), 2) for _ in range(n_samples)]
    judge_scores  = [min(1.0, h + random.uniform(-0.08, 0.08)) for h in human_scores]

    reliability = compute_inter_rater_reliability(judge_scores, human_scores)
    print(f"\nJudge vs 人工評審一致性（n={n_samples}）：")
    print(f"  Pearson r     = {reliability['pearson_r']}")
    print(f"  平均絕對誤差  = {reliability['mae']}")
    print(f"  方向一致率    = {reliability['direction_agreement']:.0%}")
    trust = "✅ 可信任 LLM Judge" if reliability['trust_judge'] else "❌ Judge 需校正"
    print(f"  結論：{trust}")

asyncio.run(run_judge_demo())

## 決策 2：Golden Dataset 版本控制

**❌ 被拒絕的方案：每次手動挑測試案例**  
- 每次評估用不同案例，無法比較版本間的差異（本次 90% 可能是比上次簡單的案例）
- 測試案例沒有難度標籤，不知道是哪類型的合約最容易出問題
- 人為選擇有偏差（傾向選「代表性」案例，忽略邊界情況）

**✅ 選擇的方案：Git 管理的 Golden Dataset**
- 每個案例有固定 `id`（SHA-based），保證跨版本可比較
- 元資料包含：難度（easy/medium/hard）、條款類型、合約類別
- 版本快照：每次模型更新都針對同一份 Dataset 評估
- 難度分層：確保測試集不會偏向簡單案例（最少 30% hard cases）

**Dataset 結構：**
```jsonl
{"id": "sha256:abc123", "input": "...", "expected_output": "...",
 "metadata": {"difficulty": "hard", "clause_type": "termination",
               "contract_category": "NDA", "added_by": "lawyer_A",
               "added_date": "2025-01-15"}}
```

In [ ]:
@dataclass
class GoldenCase:
    id: str
    input: str
    expected_output: str
    difficulty: str        # easy / medium / hard
    clause_type: str       # payment / termination / liability / ip / confidentiality
    contract_category: str # NDA / service / employment / lease / procurement
    added_by: str
    added_date: str


class GoldenDataset:
    """
    版本控制的 Golden Dataset

    核心設計：
    1. 每個案例的 id 由內容 SHA-256 決定，內容不變則 id 不變
    2. 儲存為 JSONL（每行一個案例），適合 git diff
    3. 難度分層：至少 30% hard cases
    4. 新增案例需要律師審核（PR review 流程）
    """

    VERSION = "v2.3.1"  # Semantic versioning

    def __init__(self):
        self.cases: list[GoldenCase] = []
        self._load_sample_data()

    def _make_id(self, input_text: str) -> str:
        return "sha256:" + hashlib.sha256(input_text.encode()).hexdigest()[:12]

    def _load_sample_data(self):
        """載入示範資料（實際環境從 golden_dataset.jsonl 讀取）"""
        sample_cases = [
            ("合約第5條：本合約自簽署日起生效，期限為一年，期滿自動續約，除非任一方於期滿前60日以書面通知終止。",
             "【終止條款分析】\n自動續約條款：高風險。需於期滿前60日主動通知，建議設定行事曆提醒。\n風險等級：高。",
             "hard", "termination", "service"),
            ("第2條 付款：買方應於收到發票後30日內完成付款，逾期按月息1.5%計算違約金。",
             "【付款條款分析】\n付款期限：30日（合理）。\n逾期利息：1.5%/月（年化18%），高於一般市場水準，建議議價。\n風險等級：中。",
             "medium", "payment", "procurement"),
            ("雙方同意本合約之所有資訊均屬機密，不得對外揭露。",
             "【保密條款分析】\n缺失：未定義機密資訊範圍、未設定保密期限、未列明例外情況（如法院命令）。\n建議：全面修訂保密條款。\n風險等級：高。",
             "hard", "confidentiality", "NDA"),
            ("第8條：本合約由中華民國法律管轄，爭議由台北地方法院管轄。",
             "【準據法與管轄條款】\n準據法：中華民國法律（合理）。\n管轄：台北地方法院（合理，建議確認雙方營業地點）。\n風險等級：低。",
             "easy", "jurisdiction", "service"),
            ("甲方得隨時以書面終止合約，無需說明理由。乙方如有違反，甲方得請求損害賠償但以合約金額50%為上限。",
             "【不對等條款警示】\n重大風險：甲方單方面無條件終止權（不公平條款）。\n損賠上限：50% cap 對乙方不利。\n建議：應增加對等保護條款，或要求甲方亦受相同限制。\n風險等級：高（建議律師重新審閱）。",
             "hard", "liability", "service"),
            ("員工工作期間產生之所有智慧財產權，均歸屬於公司所有。",
             "【智慧財產權條款】\n IP 歸屬明確。\n注意：範圍過廣，包含員工個人時間開發的作品。\n建議：釐清『工作期間』定義，是否限於職務範圍內。\n風險等級：中。",
             "medium", "ip", "employment"),
        ]

        for input_text, expected, difficulty, clause_type, category in sample_cases:
            self.cases.append(GoldenCase(
                id=self._make_id(input_text),
                input=input_text,
                expected_output=expected,
                difficulty=difficulty,
                clause_type=clause_type,
                contract_category=category,
                added_by="lawyer_review_panel",
                added_date="2025-01-15",
            ))

    def coverage_report(self) -> dict:
        """檢查測試集覆蓋率，確保難度分層達標"""
        total = len(self.cases)
        by_difficulty = {"easy": 0, "medium": 0, "hard": 0}
        by_clause = {}
        by_category = {}

        for c in self.cases:
            by_difficulty[c.difficulty] = by_difficulty.get(c.difficulty, 0) + 1
            by_clause[c.clause_type] = by_clause.get(c.clause_type, 0) + 1
            by_category[c.contract_category] = by_category.get(c.contract_category, 0) + 1

        hard_ratio = by_difficulty.get("hard", 0) / total
        return {
            "version": self.VERSION,
            "total_cases": total,
            "by_difficulty": by_difficulty,
            "by_clause_type": by_clause,
            "by_contract_category": by_category,
            "hard_ratio": round(hard_ratio, 2),
            "hard_ratio_ok": hard_ratio >= 0.30,
        }

    def stratified_sample(self, n: int) -> list["GoldenCase"]:
        """難度分層抽樣（確保 easy/medium/hard 各有代表）"""
        by_diff = {d: [c for c in self.cases if c.difficulty == d]
                   for d in ["easy", "medium", "hard"]}
        per_bucket = max(1, n // 3)
        sample = []
        for diff, cases in by_diff.items():
            sample.extend(random.sample(cases, min(per_bucket, len(cases))))
        return sample[:n]


# ── 示範 ──────────────────────────────────────────────────────
dataset = GoldenDataset()
report = dataset.coverage_report()

print(f"Golden Dataset 版本：{report['version']}")
print(f"總案例數：{report['total_cases']}")
print(f"\n難度分布：")
for diff, cnt in report["by_difficulty"].items():
    bar = "█" * cnt
    print(f"  {diff:<8} {bar:<15} {cnt} 案例 ({cnt/report['total_cases']:.0%})")

hard_status = "✅" if report["hard_ratio_ok"] else "❌"
print(f"\n  {hard_status} Hard case 比例：{report['hard_ratio']:.0%}（要求 >= 30%）")

print(f"\n條款類型覆蓋：{list(report['by_clause_type'].keys())}")
print(f"合約類別覆蓋：{list(report['by_contract_category'].keys())}")

sample = dataset.stratified_sample(3)
print(f"\n分層抽樣示範（n=3）：")
for c in sample:
    print(f"  [{c.difficulty:<6}] {c.id} | {c.clause_type} | {c.input[:50]}...")

## 決策 3：Shadow Mode vs. 直接 A/B 測試

**❌ 被拒絕的方案：直接上線 A/B 測試**  
法律合約審查的錯誤代價極高：
- 漏掉自動續約條款 → 客戶被迫簽 1 年不想要的合約
- 誤判高風險條款為低風險 → 客戶在未知風險下簽約
- 一旦合約簽署，損害已造成，無法回滾

**✅ 選擇的方案：Shadow Mode**
- 新模型「靜默」執行：接收相同輸入，產出結果但**不顯示給用戶**
- 收集 N 份真實合約的雙模型輸出，讓 LLM-as-Judge 離線比較
- 達到統計顯著性（win rate 差異 > 5%，p < 0.05）後才上線
- 所有比較在 24-72 小時內完成（法遵要求的 24 小時驗證窗口）

**Shadow Mode 架構：**
```
用戶請求 ──┬──→ 當前模型 ──→ 回應用戶
           │
           └──→ 候選模型 ──→ 記錄到 Shadow Log（不給用戶看）
                               │
                               └──→ 離線 Judge 比較 ──→ 決策報告
```

In [ ]:
@dataclass
class ShadowRecord:
    case_id: str
    input_hash: str
    current_output: str
    candidate_output: str
    current_score: Optional[float] = None
    candidate_score: Optional[float] = None
    winner: Optional[str] = None   # "current" / "candidate" / "tie"
    latency_current_ms: float = 0.0
    latency_candidate_ms: float = 0.0


class ShadowModeRunner:
    """
    Shadow Mode 執行器：新模型靜默並行，不影響線上結果

    法律文件場景特別重要：
    - 不讓未驗證的模型輸出直接到用戶
    - 收集足夠樣本後再做統計比較
    - 支援按合約類別分層分析
    """

    def __init__(self):
        self.records: list[ShadowRecord] = []

    async def run_shadow(
        self,
        case_id: str,
        contract_input: str,
        current_model_fn,
        candidate_model_fn,
    ) -> ShadowRecord:
        """並行執行兩個模型，候選模型結果不回傳給用戶"""
        t0 = time.monotonic()
        current_output = await current_model_fn(contract_input)
        latency_current = (time.monotonic() - t0) * 1000

        t1 = time.monotonic()
        candidate_output = await candidate_model_fn(contract_input)
        latency_candidate = (time.monotonic() - t1) * 1000

        record = ShadowRecord(
            case_id=case_id,
            input_hash=hashlib.sha256(contract_input.encode()).hexdigest()[:8],
            current_output=current_output,
            candidate_output=candidate_output,
            latency_current_ms=round(latency_current, 1),
            latency_candidate_ms=round(latency_candidate, 1),
        )
        self.records.append(record)
        return record


class ShadowComparison:
    """
    統計分析 Shadow Mode 的結果：
    win/loss/tie 分析 + 統計顯著性檢驗
    """

    def __init__(self, records: list[ShadowRecord]):
        self.records = records

    def analyze(self) -> dict:
        n = len(self.records)
        wins = sum(1 for r in self.records if r.winner == "candidate")
        losses = sum(1 for r in self.records if r.winner == "current")
        ties = sum(1 for r in self.records if r.winner == "tie")

        win_rate = wins / n if n > 0 else 0.0
        agreement_rate = ties / n if n > 0 else 0.0

        # 平均分數比較
        scored = [r for r in self.records if r.current_score and r.candidate_score]
        avg_current   = sum(r.current_score for r in scored) / len(scored) if scored else 0.0
        avg_candidate = sum(r.candidate_score for r in scored) / len(scored) if scored else 0.0

        # 延遲比較
        avg_lat_curr = sum(r.latency_current_ms for r in self.records) / n
        avg_lat_cand = sum(r.latency_candidate_ms for r in self.records) / n

        # 是否達到顯著優勢（簡單 sign test 近似）
        # 若 win_rate > 0.55（5% 改善超過基準），且樣本 > 50，視為顯著
        significant = (win_rate > 0.55) and (n >= 50)

        recommendation = "DEPLOY" if significant and avg_candidate > avg_current else \
                         "CONTINUE_SHADOW" if n < 50 else \
                         "KEEP_CURRENT"

        return {
            "n_samples": n,
            "wins_candidate": wins,
            "losses_candidate": losses,
            "ties": ties,
            "candidate_win_rate": round(win_rate, 3),
            "agreement_rate": round(agreement_rate, 3),
            "avg_score_current": round(avg_current, 3),
            "avg_score_candidate": round(avg_candidate, 3),
            "score_improvement": round(avg_candidate - avg_current, 3),
            "avg_latency_current_ms": round(avg_lat_curr, 1),
            "avg_latency_candidate_ms": round(avg_lat_cand, 1),
            "latency_change_ms": round(avg_lat_cand - avg_lat_curr, 1),
            "statistically_significant": significant,
            "recommendation": recommendation,
        }


# ── 模擬 Shadow Mode 執行 ──────────────────────────────────────
async def current_model(contract_input: str) -> str:
    await asyncio.sleep(0.001)  # 模擬 API 延遲
    return f"[Current v1.2] 審查完成。付款條款正常，違約責任中等風險。"

async def candidate_model(contract_input: str) -> str:
    await asyncio.sleep(0.001)
    return f"[Candidate v1.3] 審查完成。付款條款正常，違約責任中等風險，另發現自動續約風險（高）。"

async def run_shadow_demo():
    runner = ShadowModeRunner()

    # 模擬 60 份合約的 Shadow 執行
    n_contracts = 60
    for i in range(n_contracts):
        contract_input = f"合約 #{1000+i}: 第3條付款，第7條違約，第12條終止..."
        record = await runner.run_shadow(
            case_id=f"contract_{1000+i}",
            contract_input=contract_input,
            current_model_fn=current_model,
            candidate_model_fn=candidate_model,
        )
        # 模擬 Judge 評分（候選模型在新版略好）
        record.current_score   = round(random.uniform(0.70, 0.82), 3)
        record.candidate_score = round(random.uniform(0.76, 0.90), 3)
        record.latency_current_ms   = round(random.uniform(800, 1200), 1)
        record.latency_candidate_ms = round(random.uniform(850, 1250), 1)
        if record.candidate_score > record.current_score + 0.03:
            record.winner = "candidate"
        elif record.current_score > record.candidate_score + 0.03:
            record.winner = "current"
        else:
            record.winner = "tie"

    comparison = ShadowComparison(runner.records)
    result = comparison.analyze()

    print("Shadow Mode 比較分析：")
    print("=" * 55)
    print(f"  樣本數：{result['n_samples']} 份合約")
    print(f"  候選模型 Win/Loss/Tie：{result['wins_candidate']}/{result['losses_candidate']}/{result['ties']}")
    print(f"  候選模型 Win Rate：{result['candidate_win_rate']:.1%}")
    print(f"  平均分數：Current={result['avg_score_current']:.3f}  Candidate={result['avg_score_candidate']:.3f}")
    print(f"  分數提升：{result['score_improvement']:+.3f}")
    print(f"  延遲變化：{result['latency_change_ms']:+.1f} ms")
    sig_icon = "✅" if result['statistically_significant'] else "⏳"
    print(f"  {sig_icon} 統計顯著：{result['statistically_significant']}")

    rec_map = {"DEPLOY": "✅ 建議上線", "CONTINUE_SHADOW": "⏳ 繼續收集", "KEEP_CURRENT": "❌ 維持現狀"}
    print(f"  決策：{rec_map[result['recommendation']]}")

asyncio.run(run_shadow_demo())

## 決策 4：LLM-native Metrics vs. BLEU/ROUGE

**❌ 被拒絕的方案：只看 BLEU/ROUGE**  
BLEU/ROUGE 衡量**文字 n-gram 重疊率**，在合約審查場景完全失效：

| 問題 | 說明 |
|------|------|
| 核心問題不是「像不像」 | 合約審查的核心是「有沒有漏掉關鍵條款」，文字完全不同也可以是對的 |
| 冗長輸出得高分 | 輸出越長、詞彙越多，BLEU 越高，但可能充滿廢話 |
| 法律同義詞 | 「終止合約」和「解除契約」語意相同，BLEU 認為完全不同 |
| 無法偵測幻覺 | 捏造條款的輸出可能有高 BLEU（文字結構相似）|

**✅ 選擇的方案：Task-specific + LLM-native Metrics**

| Metric | 定義 | 目標閾值 |
|--------|------|----------|
| `clause_recall` | 識別出的關鍵條款 / 實際存在的關鍵條款 | ≥ 0.90 |
| `risk_precision` | 正確識別的高風險條款 / 所有被標為高風險的 | ≥ 0.85 |
| `hallucination_rate` | 輸出中捏造條款的比例 | ≤ 0.02 |
| `cost_per_review` | 每份合約審查的 token 費用 | ≤ $0.05 |
| `latency_p95` | 95th percentile 回應時間 | ≤ 10 秒 |
| `token_efficiency` | 有效輸出 token / 總輸出 token | ≥ 0.70 |

In [ ]:
@dataclass
class EvaluationMetrics:
    # Task-specific metrics
    clause_recall: float        # 關鍵條款識別率
    risk_precision: float       # 風險識別精確率
    hallucination_rate: float   # 幻覺率

    # LLM-native metrics
    cost_per_review_usd: float  # 每份合約成本
    latency_p50_ms: float       # 中位延遲
    latency_p95_ms: float       # 95th percentile 延遲
    token_efficiency: float     # 有效 token 比例

    # 通過閾值？
    @property
    def passes_quality_gate(self) -> bool:
        return (
            self.clause_recall >= 0.90
            and self.risk_precision >= 0.85
            and self.hallucination_rate <= 0.02
            and self.latency_p95_ms <= 10_000
            and self.cost_per_review_usd <= 0.05
        )


def compute_task_metrics(
    agent_outputs: list[str],
    reference_clauses: list[list[str]],
    reference_risks: list[list[str]],
) -> dict:
    """
    計算 Task-specific 指標

    clause_recall  = 在輸出中找到的關鍵條款 / 實際應找到的關鍵條款
    risk_precision = 正確的高風險標記 / 所有高風險標記
    hallucination  = 輸出中不存在於原文的條款描述比例（用 LLM 判斷，此處用啟發式近似）
    """
    assert len(agent_outputs) == len(reference_clauses) == len(reference_risks)
    n = len(agent_outputs)

    recall_scores, precision_scores, halluc_flags = [], [], []

    for output, clauses, risks in zip(agent_outputs, reference_clauses, reference_risks):
        out_lower = output.lower()

        # Clause Recall：用關鍵詞匹配近似
        found = sum(1 for clause_kw in clauses if clause_kw.lower() in out_lower)
        recall_scores.append(found / len(clauses) if clauses else 1.0)

        # Risk Precision：高風險標籤是否出現
        risk_kw = ["高風險", "high risk", "嚴重", "重大"]
        labeled_high = any(kw in out_lower for kw in risk_kw)
        correct_high = len(risks) > 0  # 參考答案有風險條款
        precision_scores.append(1.0 if labeled_high == correct_high else 0.0)

        # Hallucination：輸出中含有不屬於法律常用詞的虛構內容（啟發式）
        halluc_keywords = ["第99條", "法律規定", "依據台灣民法第XXX"]
        halluc_flags.append(any(kw in output for kw in halluc_keywords))

    return {
        "clause_recall": round(sum(recall_scores) / n, 4),
        "risk_precision": round(sum(precision_scores) / n, 4),
        "hallucination_rate": round(sum(halluc_flags) / n, 4),
    }


def compute_llm_native_metrics(latencies_ms: list[float],
                                 input_tokens: list[int],
                                 output_tokens: list[int],
                                 price_per_1k_output: float = 0.015) -> dict:
    """計算 LLM-native 指標（成本、延遲、Token 效率）"""
    n = len(latencies_ms)
    sorted_lat = sorted(latencies_ms)
    p50_ms = sorted_lat[int(n * 0.50)]
    p95_ms = sorted_lat[min(int(n * 0.95), n - 1)]

    total_out_tokens = sum(output_tokens)
    # Token 效率：假設有效 token = output token（實際應排除冗餘重複）
    # 用「輸出/輸入+輸出」比例作為效率近似
    efficiencies = [
        o / (i + o) if (i + o) > 0 else 0
        for i, o in zip(input_tokens, output_tokens)
    ]
    token_efficiency = sum(efficiencies) / n

    cost_per_review = (sum(output_tokens) / n / 1000) * price_per_1k_output

    return {
        "cost_per_review_usd": round(cost_per_review, 5),
        "latency_p50_ms": round(p50_ms, 1),
        "latency_p95_ms": round(p95_ms, 1),
        "token_efficiency": round(token_efficiency, 4),
    }


@dataclass
class EvalReport:
    model_version: str
    dataset_version: str
    n_cases: int
    metrics: EvaluationMetrics
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())

    def summary(self) -> str:
        m = self.metrics
        gate = "✅ PASS" if m.passes_quality_gate else "❌ FAIL"
        lines = [
            f"\n評估報告：{self.model_version} on {self.dataset_version}",
            f"測試案例：{self.n_cases} | 時間：{self.timestamp[:19]}",
            "=" * 55,
            "Task-specific Metrics:",
            f"  clause_recall       = {m.clause_recall:.4f}  (目標 ≥ 0.90) {'✅' if m.clause_recall >= 0.90 else '❌'}",
            f"  risk_precision      = {m.risk_precision:.4f}  (目標 ≥ 0.85) {'✅' if m.risk_precision >= 0.85 else '❌'}",
            f"  hallucination_rate  = {m.hallucination_rate:.4f}  (目標 ≤ 0.02) {'✅' if m.hallucination_rate <= 0.02 else '❌'}",
            "LLM-native Metrics:",
            f"  cost_per_review     = ${m.cost_per_review_usd:.4f}  (目標 ≤ $0.05) {'✅' if m.cost_per_review_usd <= 0.05 else '❌'}",
            f"  latency_p50         = {m.latency_p50_ms:.0f} ms",
            f"  latency_p95         = {m.latency_p95_ms:.0f} ms  (目標 ≤ 10,000 ms) {'✅' if m.latency_p95_ms <= 10000 else '❌'}",
            f"  token_efficiency    = {m.token_efficiency:.4f}  (目標 ≥ 0.70) {'✅' if m.token_efficiency >= 0.70 else '❌'}",
            "=" * 55,
            f"Quality Gate：{gate}",
        ]
        return "\n".join(lines)


# ── 模擬評估跑一遍 ─────────────────────────────────────────────
n_eval = 50

agent_outputs_sim = [
    f"付款條款正常。違約責任中等風險（高風險）。自動續約條款存在。" if random.random() > 0.1
    else f"本合約整體尚可。"
    for _ in range(n_eval)
]
reference_clauses_sim = [["付款", "違約", "終止"]] * n_eval
reference_risks_sim   = [["自動續約"]] * n_eval

task_m = compute_task_metrics(agent_outputs_sim, reference_clauses_sim, reference_risks_sim)

latencies_sim      = [random.uniform(1500, 8000) for _ in range(n_eval)]
input_tokens_sim   = [random.randint(800, 1500) for _ in range(n_eval)]
output_tokens_sim  = [random.randint(200, 600) for _ in range(n_eval)]

llm_m = compute_llm_native_metrics(latencies_sim, input_tokens_sim, output_tokens_sim)

metrics = EvaluationMetrics(**task_m, **llm_m)
report = EvalReport(
    model_version="contract-review-v1.3",
    dataset_version=GoldenDataset.VERSION,
    n_cases=n_eval,
    metrics=metrics,
)
print(report.summary())

## 決策 5：CI/CD 自動化迴歸測試

**❌ 被拒絕的方案：手動執行評估**  
- 工程師在趕 deadline 時很容易跳過評估（「先上線再說」）
- 沒有人監督的流程，準確率在 3 個月內悄悄從 91% 滑落到 83%，直到客訴才發現
- 不同人執行時使用不同的測試案例，結果無法比較

**✅ 選擇的方案：PR merge 前自動觸發**  
- 任何 model prompt 或 pipeline 改動的 PR，都會自動跑 Regression Gate
- 若 `clause_recall` 下降 > 5% 或 `hallucination_rate` > 2%，**自動阻止 merge**
- 在 PR comment 自動貼上對比報告，工程師看到數字才能決定要不要 override

**CI/CD 流程：**
```
git push PR ──→ GitHub Actions ──→ [Regression Gate]
                                         │
               ┌─────────────────────────┤
               │  1. 載入 baseline 指標（main branch）
               │  2. 在 Golden Dataset 跑新版本
               │  3. 比較指標差異
               │  4. 產生 PR comment
               └──→ Pass: merge allowed
               └──→ Fail: merge blocked (status check failed)
```

In [ ]:
@dataclass
class RegressionThresholds:
    """可設定的迴歸門檻（法遵部門審核後設定）"""
    max_clause_recall_drop: float    = 0.05   # 最多允許下降 5%
    max_risk_precision_drop: float   = 0.05
    max_hallucination_rate: float    = 0.02   # 幻覺率絕對上限 2%
    max_latency_p95_increase_ms: float = 2000 # P95 延遲最多增加 2 秒
    max_cost_increase_ratio: float   = 0.20   # 成本最多增加 20%


class RegressionGate:
    """
    CI/CD 迴歸門檻：比較 PR 分支 vs main branch 的評估指標
    自動產生 PR comment，並決定是否阻止 merge
    """

    def __init__(self, thresholds: Optional[RegressionThresholds] = None):
        self.thresholds = thresholds or RegressionThresholds()

    def compare_to_baseline(
        self,
        baseline: EvaluationMetrics,
        candidate: EvaluationMetrics,
    ) -> dict:
        """比較兩版本指標，回傳詳細的通過/失敗清單"""
        t = self.thresholds
        checks = [
            (
                "clause_recall",
                baseline.clause_recall - candidate.clause_recall,
                t.max_clause_recall_drop,
                "下降",
            ),
            (
                "risk_precision",
                baseline.risk_precision - candidate.risk_precision,
                t.max_risk_precision_drop,
                "下降",
            ),
            (
                "hallucination_rate",
                candidate.hallucination_rate,
                t.max_hallucination_rate,
                "超標",
            ),
            (
                "latency_p95",
                candidate.latency_p95_ms - baseline.latency_p95_ms,
                t.max_latency_p95_increase_ms,
                "增加",
            ),
        ]

        results = []
        all_pass = True
        for name, delta, threshold, direction in checks:
            passed = delta <= threshold
            if not passed:
                all_pass = False
            results.append({
                "metric": name,
                "delta": round(delta, 4),
                "threshold": threshold,
                "direction": direction,
                "passed": passed,
            })

        return {"all_pass": all_pass, "checks": results}

    def generate_pr_comment(
        self,
        baseline: EvaluationMetrics,
        candidate: EvaluationMetrics,
        pr_number: int,
        commit_sha: str,
    ) -> str:
        """產生貼在 GitHub PR 的 Markdown 評估報告"""
        comparison = self.compare_to_baseline(baseline, candidate)
        status = "✅ PASS — merge allowed" if comparison["all_pass"] else "❌ FAIL — merge blocked"

        def fmt_delta(name, baseline_val, candidate_val, lower_is_better=False):
            delta = candidate_val - baseline_val
            icon = "✅" if (delta >= 0) != lower_is_better else "❌"
            direction = "+" if delta >= 0 else ""
            return f"| {name} | {baseline_val:.4f} | {candidate_val:.4f} | {direction}{delta:.4f} | {icon} |"

        lines = [
            f"## 🤖 AI 品質迴歸測試 — PR #{pr_number}",
            f"**Commit:** `{commit_sha}`  ",
            f"**Dataset:** `{GoldenDataset.VERSION}`  ",
            f"**結論：{status}**",
            "",
            "### Task-specific Metrics",
            "| Metric | Baseline | Candidate | Delta | Status |",
            "|--------|----------|-----------|-------|--------|",
            fmt_delta("clause_recall",      baseline.clause_recall,      candidate.clause_recall),
            fmt_delta("risk_precision",     baseline.risk_precision,     candidate.risk_precision),
            fmt_delta("hallucination_rate", baseline.hallucination_rate, candidate.hallucination_rate, lower_is_better=True),
            "",
            "### LLM-native Metrics",
            "| Metric | Baseline | Candidate | Delta | Status |",
            "|--------|----------|-----------|-------|--------|",
            fmt_delta("cost_per_review ($)", baseline.cost_per_review_usd, candidate.cost_per_review_usd, lower_is_better=True),
            fmt_delta("latency_p95 (ms)",    baseline.latency_p95_ms,    candidate.latency_p95_ms,    lower_is_better=True),
            fmt_delta("token_efficiency",    baseline.token_efficiency,   candidate.token_efficiency),
        ]

        if not comparison["all_pass"]:
            lines.append("\n### ❌ 失敗項目說明")
            for check in comparison["checks"]:
                if not check["passed"]:
                    lines.append(
                        f"- `{check['metric']}` {check['direction']} {check['delta']:.4f}，"
                        f"超過閾值 {check['threshold']}"
                    )

        lines.append("\n*此評估由 CI/CD Pipeline 自動產生，覆寫需要 Legal Reviewer 手動 approve*")
        return "\n".join(lines)


# ── 示範：模擬一個讓品質下滑的 PR ────────────────────────────
baseline_metrics = EvaluationMetrics(
    clause_recall=0.921, risk_precision=0.893, hallucination_rate=0.008,
    cost_per_review_usd=0.032, latency_p50_ms=2100, latency_p95_ms=6800,
    token_efficiency=0.748,
)

# PR 場景 1：只優化成本，但品質下滑
bad_pr_metrics = EvaluationMetrics(
    clause_recall=0.851,    # 下降 7%（超過 5% 門檻）
    risk_precision=0.870,
    hallucination_rate=0.025,  # 幻覺率 2.5%（超過 2% 上限）
    cost_per_review_usd=0.019,  # 成本降低，但品質不行
    latency_p50_ms=1800,
    latency_p95_ms=5200,
    token_efficiency=0.791,
)

# PR 場景 2：品質提升
good_pr_metrics = EvaluationMetrics(
    clause_recall=0.943,
    risk_precision=0.912,
    hallucination_rate=0.005,
    cost_per_review_usd=0.035,
    latency_p50_ms=2300,
    latency_p95_ms=7100,
    token_efficiency=0.762,
)

gate = RegressionGate()

print("=" * 55)
print("PR #142：削減成本版本（預期阻止 merge）")
print("=" * 55)
print(gate.generate_pr_comment(baseline_metrics, bad_pr_metrics,
                                pr_number=142, commit_sha="a3f9c12"))

print("\n" + "=" * 55)
print("PR #143：品質優化版本（預期允許 merge）")
print("=" * 55)
print(gate.generate_pr_comment(baseline_metrics, good_pr_metrics,
                                pr_number=143, commit_sha="b7d2e45"))

## 統計分析：如何讓數字說話

法遵部門問：「你說準確率 90%，但這個數字可信嗎？樣本夠大嗎？」  
以下三個統計工具，讓你能精確回答這個問題。

In [ ]:
import math

# ────────────────────────────────────────────
# 1. 準確率的信賴區間（Wilson Score Interval）
# ────────────────────────────────────────────
def wilson_confidence_interval(successes: int, n: int,
                                confidence: float = 0.95) -> tuple[float, float]:
    """
    Wilson Score Interval：比 Normal approximation 在小樣本更準確
    適用：binary metric（pass/fail，是否識別關鍵條款）

    公式：
      center = (p_hat + z²/2n) / (1 + z²/n)
      margin = z * sqrt(p_hat*(1-p_hat)/n + z²/(4n²)) / (1 + z²/n)
    """
    z = 1.96 if confidence == 0.95 else 2.576  # 95% 或 99% CI
    p_hat = successes / n if n > 0 else 0.0
    z2 = z * z
    center = (p_hat + z2 / (2 * n)) / (1 + z2 / n)
    margin = z * math.sqrt(p_hat * (1 - p_hat) / n + z2 / (4 * n * n)) / (1 + z2 / n)
    return (max(0.0, round(center - margin, 4)),
            min(1.0, round(center + margin, 4)))


# ────────────────────────────────────────────
# 2. 兩模型比較的統計顯著性（Bootstrap Test）
# ────────────────────────────────────────────
def bootstrap_significance_test(
    scores_a: list[float],
    scores_b: list[float],
    n_bootstrap: int = 2000,
    alpha: float = 0.05,
) -> dict:
    """
    Bootstrap Test：適合非常態分佈的 LLM 指標（不假設常態）

    方法：
    1. 計算觀察到的差異 d_obs = mean(B) - mean(A)
    2. 重複抽樣 2000 次，每次計算 mean(B*) - mean(A*)
    3. p-value = 觀察差異在抽樣分佈中的位置
    """
    n = min(len(scores_a), len(scores_b))
    mean_a = sum(scores_a[:n]) / n
    mean_b = sum(scores_b[:n]) / n
    d_obs = mean_b - mean_a

    # Bootstrap 重抽樣
    count_extreme = 0
    pooled = scores_a[:n] + scores_b[:n]
    for _ in range(n_bootstrap):
        sample_a = [random.choice(scores_a[:n]) for _ in range(n)]
        sample_b = [random.choice(scores_b[:n]) for _ in range(n)]
        d_boot = sum(sample_b) / n - sum(sample_a) / n
        if abs(d_boot - (sum(pooled) / len(pooled) - sum(pooled) / len(pooled))) >= abs(d_obs):
            count_extreme += 1

    p_value = count_extreme / n_bootstrap
    is_significant = p_value < alpha

    return {
        "mean_a": round(mean_a, 4),
        "mean_b": round(mean_b, 4),
        "observed_diff": round(d_obs, 4),
        "p_value": round(p_value, 4),
        "significant": is_significant,
        "conclusion": f"B {'顯著優於' if (is_significant and d_obs > 0) else '顯著差於' if (is_significant and d_obs < 0) else '與'} A (p={p_value:.3f})",
    }


# ────────────────────────────────────────────
# 3. 所需樣本數計算
# ────────────────────────────────────────────
def required_sample_size(
    baseline_rate: float = 0.90,
    minimum_detectable_effect: float = 0.05,
    alpha: float = 0.05,
    power: float = 0.80,
) -> int:
    """
    計算偵測指定效果所需的最小樣本數

    公式（two-proportion z-test）：
      n = (z_alpha/2 + z_beta)² × (p1(1-p1) + p2(1-p2)) / (p1-p2)²

    場景：偵測 clause_recall 從 90% 下滑到 85%（5% drop）
    需要多少測試案例才能以 95% 信心偵測到？
    """
    p1 = baseline_rate
    p2 = baseline_rate - minimum_detectable_effect

    z_alpha = 1.96   # 95% confidence (two-tailed)
    z_beta  = 0.842  # 80% power
    if power >= 0.90:
        z_beta = 1.282

    numerator = (z_alpha + z_beta) ** 2 * (p1 * (1 - p1) + p2 * (1 - p2))
    denominator = (p1 - p2) ** 2
    n = math.ceil(numerator / denominator)
    return n


# ── 示範 ──────────────────────────────────────────────────────
print("統計分析工具示範")
print("=" * 55)

# 1. 信賴區間
print("\n1. 準確率信賴區間（Wilson Score）：")
scenarios = [
    (45, 50,  "50 個案例，45 個正確"),
    (90, 100, "100 個案例，90 個正確"),
    (225, 250, "250 個案例，225 個正確"),
    (450, 500, "500 個案例，450 個正確"),
]
for successes, n, desc in scenarios:
    lo, hi = wilson_confidence_interval(successes, n)
    p_hat = successes / n
    width = hi - lo
    print(f"  {desc}")
    print(f"    觀測準確率：{p_hat:.0%}，95% CI = [{lo:.3f}, {hi:.3f}]（寬度 {width:.3f}）")

# 2. 模型 A/B 比較
print("\n2. 兩模型統計顯著性比較（Bootstrap, n=100）：")
scores_model_a = [random.uniform(0.75, 0.92) for _ in range(100)]
scores_model_b = [s + random.uniform(0.02, 0.07) for s in scores_model_a]  # B 系統性略好
sig_result = bootstrap_significance_test(scores_model_a, scores_model_b, n_bootstrap=1000)
print(f"  Model A 平均：{sig_result['mean_a']:.4f}")
print(f"  Model B 平均：{sig_result['mean_b']:.4f}")
print(f"  差異：{sig_result['observed_diff']:+.4f}")
print(f"  p-value：{sig_result['p_value']:.4f}")
print(f"  結論：{sig_result['conclusion']}")

# 3. 樣本數計算
print("\n3. 所需樣本數計算：")
for mde, power in [(0.05, 0.80), (0.05, 0.90), (0.03, 0.80), (0.02, 0.80)]:
    n = required_sample_size(
        baseline_rate=0.90,
        minimum_detectable_effect=mde,
        power=power,
    )
    print(f"  偵測 {mde:.0%} 下滑（{power:.0%} power）：需要 {n} 個測試案例")

print("\n  → 法遵要求：能偵測 5% 下滑，需要 Golden Dataset 至少 200 個案例")

## FDE 面試白板語言

### 常見面試題 Q&A

In [ ]:
print("""
FDE 面試：Agent 統計評估框架的核心問題
═══════════════════════════════════════════════════════

Q: 怎麼設計 LLM-as-Judge 避免它偏袒自己的輸出？
A: 三個設計原則：
   1. 用不同模型當 Judge：如果被評估的是 Gemini Pro，
      就用 GPT-4o 當 Judge，反之亦然。研究顯示
      跨模型 Judge 的 self-preference bias 接近零。
   2. 隨機化答案順序：每次評估隨機決定哪個答案在前，
      避免 position bias（LLM 傾向給第一個答案高分）。
   3. 要求結構化評分 + 說明：Judge 必須給出
      score + reasoning + specific_issues，
      不能只回 7 分。這讓評分過程可解釋，也更容易
      發現 Judge 是否在說廢話。
   驗證方式：用 100 個有人工標籤的案例，計算
   Pearson r > 0.85 才信任這個 Judge。

Q: Shadow Mode 要跑多久才能有統計顯著性？
A: 取決於你要偵測多大的差異。
   「偵測 5% 的 clause_recall 差異，95% 信心度，80% power」
   需要約 200 個合約樣本（用 Wilson CI 公式計算）。
   法律事務所每天審查約 250 份，所以 1 天的流量就夠了。
   但如果你想偵測 2% 的細微差異，需要約 1,200 個樣本，
   約需 5 天。所以 Shadow Mode 的「等待期」不是固定的，
   要根據你設定的 MDE（Minimum Detectable Effect）決定。

Q: 為什麼 BLEU/ROUGE 不適合評估 Agent？
A: BLEU/ROUGE 量的是「文字 n-gram 重疊」，
   但合約審查 Agent 的核心問題是「有沒有發現關鍵風險」。
   三個具體失敗案例：
   1. 一份正確的審查報告，用不同的法律用語表達，
      BLEU 可能只有 0.3（被認為很差），但實質完全正確。
   2. 一份充滿廢話但文字和參考答案很像的報告，
      BLEU 可能有 0.7（被認為很好），但漏掉了高風險條款。
   3. Hallucination（捏造條款）的輸出，
      文字結構和參考答案很像，BLEU 不會懲罰它。
   Task-specific metrics 才能真正衡量「AI 幫律師找到了什麼」。

Q: 客戶問「你的 Agent 準確率多少？」你怎麼回答？
A: 不能只說一個數字。正確回答方式：
   「在我們的 250 個案例 Golden Dataset（涵蓋 NDA、
   服務合約、勞動合約，30% 為高難度案例）上，
   clause_recall 是 92.1%（95% CI：[89.3%, 94.5%]），
   risk_precision 是 88.7%，幻覺率是 0.8%。
   這個數字是基於和貴所現有案例同分布的測試集，
   若有特殊合約類型可以擴充測試集後重新評估。」
   重點：準確率 + 信賴區間 + 測試集描述 + 幻覺率。
   法遵部門最在意的是「幻覺率」和「漏判高風險」，
   不是整體準確率。
═══════════════════════════════════════════════════════
""")